In [1]:
!pip install --upgrade torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.1/797.1 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
import argparse
import os
import random,numpy
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [3]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [4]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

workers = 2
batch_size = 10
nz = 100
num_epochs = 5
lr = 0.001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

Random Seed:  999


In [5]:
transform = transforms.Compose(
    [transforms.Resize(256),transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.ImageFolder(root=f'/kaggle/input/rsna-data/train_wolf/right_neural_foraminal_narrowing',transform=transform)
dataloader = torch.utils.data.DataLoader(trainset,batch_size=batch_size,shuffle=True,num_workers=2)

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [6]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.devil=nn.Sequential(
            nn.Conv2d(3, 4,3),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(4, 8, 3),
            nn.BatchNorm2d(8),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(8, 16, 3),
            nn.BatchNorm2d(16),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.AvgPool2d(2, 2),

            nn.Flatten(),

            nn.Linear(512, 264),
            nn.ReLU(),
            nn.Linear(264, 132),
            nn.ReLU(),
            nn.Linear(132, 64),
            nn.ReLU(),
            nn.Linear(64, 15)
        )

    def forward(self,x):
        return self.devil(x)


In [7]:
netD = Discriminator().to(device)
if (device.type == 'cuda') and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))
netD.apply(weights_init)

unique, counts = np.unique(np.array(trainset.targets), return_counts=True)
print("Unique:",unique,counts)
class_counts = dict(zip(unique, counts))
weights=[]
for i in range(len(dataloader.dataset.classes)):
    weights.append(max(class_counts.values())/class_counts[i])  
class_weights = torch.FloatTensor(weights).to(device) 
print(weights)
criterion,img_devil = nn.CrossEntropyLoss(weight=class_weights),0
fixed_noise = torch.randn(1, nz, 1, 1, device=device)

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
schedulerD=torch.optim.lr_scheduler.ExponentialLR(optimizerD, gamma=0.86)

Unique: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14] [  63   13 1890  167    6 1793  414   39 1512  628  130 1208  495  190
 1281]
[30.0, 145.3846153846154, 1.0, 11.317365269461078, 315.0, 1.0540992749581706, 4.565217391304348, 48.46153846153846, 1.25, 3.0095541401273884, 14.538461538461538, 1.564569536423841, 3.8181818181818183, 9.947368421052632, 1.4754098360655739]


In [8]:
os.mkdir(f"/kaggle/working/weight_1/")
#netD.load_state_dict(torch.load("/kaggle/input/rafire_4/pytorch/default/1/rafire.pth"))
z_w_=[]
i_w=0
while(i_w<2):
    z_,z,z_w=0,0,{}
    for i, data in enumerate(dataloader, 0):
        optimizerD.zero_grad()
        real_cpu=data[0].cuda()
        label=data[1].cuda()
        output = netD(real_cpu)
        errD_real = criterion(output,label)
        errD_real.backward()
        optimizerD.step()
        wolf_z=torch.argmax(netD(real_cpu),dim=1).view(-1)
        #print(wolf_z,label)
        for i,j in zip(wolf_z,label):
            if i==j:
                if dataloader.dataset.classes[i] not in z_w.keys():
                    z_w[dataloader.dataset.classes[i]]=1
                else:
                    z_w[dataloader.dataset.classes[i]]+=1
                z+=1
            z_+=1
    z_w_.append(z)
    
    print(z_,z)	
    for i in z_w.keys():
        print(f"{i} : {z_w[i]}")
        
    if len(z_w_)>=3:
        if len([True for i in range(1,3) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3]+2 and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]-3])==2:
            z_w_=[]
            print(optimizerD.param_groups[0]["lr"])
            schedulerD.step()
            print(optimizerD.param_groups[0]["lr"])
    torch.save(netD.state_dict(),f'/kaggle/working/weight_1/{z}.pth')
    torch.save(netD.state_dict(),f'/kaggle/working/rafire.pth')
    i_w+=1

9829 1637
right_neural_foraminal_narrowing_l5_s1_normal : 259
right_neural_foraminal_narrowing_l1_l2_normal : 190
right_neural_foraminal_narrowing_l2_l3_normal : 505
right_neural_foraminal_narrowing_l4_l5_Moderate : 103
right_neural_foraminal_narrowing_l5_s1_Moderate : 26
right_neural_foraminal_narrowing_l3_l4_normal : 196
right_neural_foraminal_narrowing_l1_l2_Moderate : 1
right_neural_foraminal_narrowing_l4_l5_normal : 291
right_neural_foraminal_narrowing_l3_l4_Moderate : 57
right_neural_foraminal_narrowing_l4_l5_Severe : 3
right_neural_foraminal_narrowing_l5_s1_Severe : 6
9829 1606
right_neural_foraminal_narrowing_l2_l3_normal : 407
right_neural_foraminal_narrowing_l5_s1_normal : 238
right_neural_foraminal_narrowing_l4_l5_normal : 73
right_neural_foraminal_narrowing_l1_l2_normal : 422
right_neural_foraminal_narrowing_l3_l4_normal : 339
right_neural_foraminal_narrowing_l5_s1_Moderate : 38
right_neural_foraminal_narrowing_l4_l5_Moderate : 67
right_neural_foraminal_narrowing_l3_l4_Mode